# 非评分实验 2：LLM 调用与构建简单增强提示词

欢迎来到 **LLM 调用与构建简单增强提示词** 实验。在本实验中，你将动手练习两个与大语言模型（LLM）交互的核心函数。这些函数既支持向 LLM 发送单轮提示，也支持进行多轮对话。本实验的主要目标是演示如何在提示词中加入额外信息，使提示更完整、更有用。这些额外上下文能够帮助模型给出更好、更精准的回答。

在本实验中，你将学习：

- 如何完成 LLM 调用设置，并分别发起单轮提问与多轮对话。
- 如何使用附加数据丰富提示词，从而提升模型回复质量。


# 目录
- [ 1 - 理解 LLM 调用函数](#1)
  - [ 1.1 `generate_with_single_input`](#1-1)
  - [ 1.2 `generate_with_multiple_input`](#1-2)
- [ 2 - 将数据整合到 LLM 提示词中](#2)
  - [ 2.1 理解数据结构](#2-1)
  - [ 2.2 创建提示词](#2-2)


In [1]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input,
    get_proxy_url,
    get_proxy_headers,
    get_together_key
)

<a id='1'></a>
## 1 - 理解 LLM 调用函数

在本节中，你将探索本课程用于调用 LLM 的函数。该函数通过 [together.ai](https://www.together.ai/) API 调用模型。在 Coursera 环境中，访问 Together API 的部分步骤会由代理服务器代为处理，因此如果你在 Coursera 环境外直接运行本 notebook，通常无法立即生效。不过，做少量调整后，你可以传入可选的 together.ai API key，从而在本地机器运行这些 notebook。

<a id='1-1'></a>
### 1.1 `generate_with_single_input`

该函数允许你基于单个输入提示从语言模型生成文本。当前先关注仅使用少量基础参数的简单调用。你将在第 4 模块进一步探索调用 LLM 的不同参数及其对输出的影响。以下是当前可用参数。

#### 参数：

- `prompt` (str)：要发送给语言模型的输入提示文本。
- `max_tokens` (int)：响应中可生成的最大 token 数。
- `model` (str)：模型名称，默认值为 `"Qwen/Qwen3.5-9B"`。
- 请求负载中通过 `reasoning={"enabled": False}` 关闭推理，以避免生成不必要的推理 token。
- `together_api_key`：用于认证的可选 API key，默认值为 `None`。若为 `None`，将使用课程提供的代理；否则将使用提供的 API key 直接调用 together.ai。

In [ ]:
# 示例调用
output = generate_with_single_input(
    prompt="What is the capital of France?"
 )

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: The capital of France is **Paris**.

Located in the north-central part of the country along the Seine River, Paris is not only the political center but also the largest city in France. It is renowned globally for its art, fashion, gastronomy, and culture, and has been the capital since the 12th century.


<a id='1-2'></a>
### 1.2 `generate_with_multiple_input`

该函数用于在对话场景中处理多条输入消息。输入格式为包含两个键的字典：

1. 'role'：消息所属角色（通常是 assistant、system 或 user）
2. 'content'：提示内容

#### 参数：

- `messages` (List[Dict])：字典列表，每个字典包含 'role' 与 'content' 两个键，用于表示对话中的一条消息。
- `max_tokens` (int)：控制响应的 token 上限。
- `model` (str)：使用的模型，默认值为 `"Qwen/Qwen3.5-9B"`。
- 与单输入辅助函数一致，默认关闭 reasoning 以避免生成不必要 token。

In [ ]:
# 示例调用
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: The captain of the French national team during the 2018 FIFA World Cup was **Hugo Lloris**.

The veteran goalkeeper led the team to their third title, playing a key role in their victory over Croatia in the final.


### 1.3 与 OpenAI 库集成

[Together.ai](together.ai) 的端点与 [OpenAI 接口兼容](https://docs.together.ai/docs/openai-api-compatibility)，因此你可以使用 [OpenAI Python 库](https://github.com/openai/openai-python) 发起调用。本节将演示具体做法。

In [4]:
from openai import OpenAI, DefaultHttpxClient
import httpx

In [ ]:
base_url = get_proxy_url() # 若使用 together 端点，请在此填写 https://api.together.xyz/

# 自定义传输以跳过 SSL 校验。仅在使用课程代理时需要；否则可忽略。
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# 使用自定义 transport 创建 DefaultHttpxClient 实例
http_client = DefaultHttpxClient(transport=transport, headers=get_proxy_headers())

client = OpenAI(
    api_key = get_together_key(), # 使用课程代理时可填任意值；若直连 together 端点，请填写 together API key。
    base_url=base_url, 
    http_client=http_client, # 通过代理调用时用于绕过 SSL；若直连 together.ai 端点可移除。
)

要使用它，我们沿用前面的同一示例。

In [6]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [7]:
response = client.chat.completions.create(messages = messages, model ="Qwen/Qwen3.5-9B", extra_body={
        "reasoning": False
    })

In [8]:
print(response)

ChatCompletion(id='ocFRNF4-6Ng1vN-9e0ff98a5e43251d', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The captain of the French national team during the 2018 FIFA World Cup was **Hugo Lloris**, the goalkeeping duo.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Current conversation: User asked about the 2018 FIFA World Cup winner, and I answered "France".\n    *   New question: "Who was the captain?"\n    *   Context: The captain of the French national football team during the 2018 FIFA World Cup.\n\n2.  **Retrieve Knowledge:**\n    *   Event: 2018 FIFA World Cup (held in Russia).\n    *   Champion: France.\n    *   Team: France national football team (Équipe de France).\n    *   Key Figure/Leader: The coach was Didier Deschamps. Who was the captain?\n    *   Captain during that tournament: Hugo Lloris.\n\n3.  **Ve

注意，返回结果包含多个属性。若要访问响应正文，可运行：response.choices[0].message.content

In [9]:
print(response.choices[0].message.content)

The captain of the French national team during the 2018 FIFA World Cup was **Hugo Lloris**, the goalkeeping duo.


### 1.4 本课程中的 Together.ai 集成说明

[Together.ai](https://together.ai) 为本课程中使用其托管 LLM 提供了慷慨的额度支持。虽然技术上存在额度上限，但该上限大约是你通常使用量的 10 倍，即使进行大量实验一般也足够。这是为了确保你有充分空间探索，获得更顺畅的学习体验。

在本课程进行 LLM 调用时，你可能遇到两类主要错误：

1. **500 与 429 错误**：当系统调用过多、负载过高时会出现，通常稍等片刻即可恢复。
   
2. 若额度确实用尽，系统会提示你。在这种情况下，请在 Discourse 社区联系课程团队。

作业评分不会消耗你的任何额度。我们希望你几乎不需要关注额度上限；即便极少数情况下触发上限，你也能快速知道原因并得到及时处理。

<a id='2'></a>
## 2 - 将数据整合到 LLM 提示词中

在本节中，你将学习如何在将提示词传给大语言模型（LLM）之前，有效地把数据整合进提示。我们会使用一个小型数据集，其中包含若干房屋信息（JSON 结构）。这将帮助你理解 RAG 场景下的提示增强方法。

<a id='2-1'></a>
### 2.1 理解数据结构

先快速看一下数据结构。这是一个很小的房屋数据集：列表中每个字典对应一套房屋。

In [10]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 创建提示词

先从构建提示词开始。第一步是为数据设计一个布局格式。

In [ ]:
# 首先，为房屋信息创建展示布局

def house_info_layout(houses):
    # 创建空字符串
    layout = ''
    # 遍历房屋列表
    for house in houses:
        # 对每套房屋，使用 f-string 将信息追加到字符串中
        # 使用括号换行拼接有助于提升可读性：每一行都可独立追加一段 f-string
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # 别忘了末尾换行符！
    return layout

In [ ]:
# 查看布局结果
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



现在创建一个函数来生成要传给大语言模型（LLM）的提示词。该函数接收用户查询和可用房屋数据作为输入，以便更有效地回答用户问题。

In [ ]:
def generate_prompt(query, houses):
    # 上面写的代码足够模块化，可接收任意房屋列表，因此你也可以只传入数据子集。
    # 这在更复杂场景中很有用：你可能只想给 LLM 一部分信息，而不是全部数据。
    houses_layout = house_info_layout(houses)
    # 创建固定模板提示。这里可用三重双引号（"""），减少单双引号冲突带来的问题。
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

In [14]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


现在来调用 LLM！

In [ ]:
query = "What is the most expensive house? And the bigger one?"
# 不带房屋信息的调用：将角色设为 user
query_without_house_info = generate_with_single_input(prompt = query, role = 'user')
# 带房屋信息的调用：按当前提示结构，将角色设为 assistant
enhanced_query = generate_prompt(query, houses = house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query, role = 'assistant')

In [ ]:
# 不带房屋信息
print(query_without_house_info['content'])

The answer depends heavily on whether you are looking for **officially completed sales** (proven deals) or **listed prices** (asking prices for houses that may never sell), and how you define "expensive" or "bigger" (total land area vs. living space).

### 1. The Most Expensive House (Verified Sales)
There is a clear distinction between the most expensive house **sold** and the most expensive house **listed**.

*   **Most Expiciously *Sold* House:**
    The title belongs to the **Beverly Hills Football Ranch** in Beverly Hills, California.
    *   **Price:** Approximately **$10.3 million** (sold in 1984).
    *   **The Catch:** While the sale price was high, the property is massive (featuring a 5-acre playground, a golf course, and multiple pools). In many metrics, it is considered the "cheapest" million-dollar mansion because the price per square foot is incredibly low due to its enormous size.
    *   **Modern Record Holder (Recent Transaction):** The **Christie's One Bignams** in Po

In [ ]:
# 带房屋信息
print(query_with_house_info['content'])

Based on the information provided:

*   **The most expensive house** is located at **456 Elm Avenue, Shelbyville, TN**. It is priced at **$320,000**, which is higher than the $230,000 price of the house on Maple Street.
*   **The bigger house** (by area) is also located at **456 Elm Avenue, Shelbyville, TN**. It has **2,500 sq ft** of space, compared to the 1,500 sq ft of the house on Maple Street.


继续加油！你已完成关于 LLM 调用与提示增强的入门非评分实验！